# SSPNet 통합 평가 · 무료 Colab

**처음에는 1→2→3→4→5 순서로 실행하세요.** 무료 T4 GPU를 선택합니다.

1. **설정:** 전체 검사(`full`)·분할 검사(`tiles`)와 데이터 범위를 한 곳에서 선택합니다.
2. **환경·가중치 준비:** 설치와 가중치 준비를 한 번에 처리합니다. 같은 실행 환경에서는 기존 파일을 재사용합니다.
3. **입력 준비:** 데이터 COCO ZIP을 업로드합니다. 전체 검사와 비교하려면 기존 전체 검사 결과 ZIP도 준비합니다.
4. **추론·평가:** 모델은 사진 전체에서 사람을 찾습니다. 동일한 예측을 전체 사람·물속 사람 기준으로 각각 채점합니다.
5. **결과·다운로드:** 두 기준의 결과와 전체/분할 비교표를 확인하고 ZIP을 받습니다.

기본값은 **AnnotationPiriapolis v4 전체 635장 / 분할 320픽셀 / 기본 겹침 20% / 점수 0.25 / IoU 0.3**입니다. IoU 0.5 참고 결과도 저장합니다. 가장자리에서는 설정한 비율보다 더 겹칠 수 있습니다.

**설정 변경 시:** 1→2→4→5를 실행합니다. 데이터·분할도 변경했다면 3단계도 다시 실행합니다. **Colab 임시 컴퓨터가 새로 배정됨:** 1→2→3→4→5를 실행합니다. 같은 설정의 중단된 추론은 저장된 완료 기록을 재사용합니다.

학습 전후 비교는 `EVALUATION_MODE=full`, `SPLIT=all`, `MAX_IMAGES=0`으로 설정합니다. 공개 가중치는 `public`, 학습 가중치는 Drive 경로를 입력하는 `trained_path` 또는 직접 올리는 `trained_upload`를 고르고 2단계를 실행합니다. Drive 경로를 사용하려면 `USE_GOOGLE_DRIVE`를 켭니다. 학습 가중치 결과는 가중치 해시별 별도 폴더에 저장됩니다.

크기 제한 없는 전체 사람 결과가 주 결과입니다. 물속 사람 결과와 `s=√(가로×세로)` 기준의 2~8·8~20픽셀 재현율은 보조 결과입니다. 물속 사람 기준에서는 모래사장 사람 탐지도 오탐에 포함됩니다. 기존 주석의 누락·위치 오류를 자동 보정하지 않습니다.


## 1. 설정

전체 635장 분할 평가에는 기본값을 유지하세요. 전체 검사 결과 ZIP과 비교하려면 `COMPARE_WITH_FULL`을 켭니다. `MAX_IMAGES=0`은 선택한 분할의 모든 이미지입니다.


In [ ]:
#@title 1. 설정
import os,sys,json,shutil,subprocess,zipfile
from pathlib import Path
from IPython.display import display,Image as DisplayImage

TASK = "dataset" #@param ["dataset", "single_image"]
EVALUATION_MODE = "tiles" #@param ["full", "tiles"]
WEIGHT_SOURCE = "public" #@param ["public", "trained_path", "trained_upload"]
TRAINED_WEIGHT_PATH = "" #@param {type:"string"}
DATASET_NAME = "AnnotationPiriapolis_v4" #@param ["AnnotationPiriapolis_v4", "YoloBeachTrain_v2", "YoloBeachTrain_v3"]
SPLIT = "all" #@param ["all", "test", "valid", "train"]
MAX_IMAGES = 0 #@param {type:"integer"}
TILE_SIZE = 320 #@param {type:"integer"}
TILE_OVERLAP = 0.20 #@param {type:"number"}
REPORT_SCORE = 0.25 #@param {type:"number"}
MATCH_IOU = 0.3 #@param {type:"number"}
COMPARE_WITH_FULL = True #@param {type:"boolean"}
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}

# 이미 Colab/Drive에 있는 파일은 아래 경로를 입력할 수 있습니다. 비어 있으면 기존 파일을 찾거나 업로드합니다.
DATASET_ZIP_PATH = ""
BASELINE_ZIP_PATH = ""
IMAGE_SOURCE = "upload" # upload / demo / path
IMAGE_PATH_SETTING = ""
WORK=Path("/content/sspnet_free")
WORK.mkdir(exist_ok=True)
CONFIG=dict(task=TASK,mode=EVALUATION_MODE,dataset_name=DATASET_NAME,split=SPLIT,max_images=MAX_IMAGES,
            tile_size=TILE_SIZE,tile_overlap=TILE_OVERLAP,report_score=REPORT_SCORE,match_iou=MATCH_IOU,
            long_edge=1280,short_edge=1024,compare_with_full=COMPARE_WITH_FULL)
CONFIG["weight_source"]=WEIGHT_SOURCE
if MAX_IMAGES<0 or TILE_SIZE<=0 or not 0<=TILE_OVERLAP<1 or not .001<=REPORT_SCORE<=1 or not 0<MATCH_IOU<=1:
    raise ValueError("이미지 수·분할 크기·점수·IoU 설정을 확인하세요.")
CONFIG_PATH=WORK/"unified_settings.json"
CONFIG_PATH.write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2),encoding="utf-8")
print("설정 저장:",TASK,EVALUATION_MODE,DATASET_NAME,SPLIT,"/ 전체 이미지" if MAX_IMAGES==0 else f"/ 최대 {MAX_IMAGES}장")


## 2. 환경·가중치 준비

처음 또는 새 실행 환경에서 실행하세요. 설치가 끝날 때까지 기다립니다. GPU 연산 검사를 통과해야 합니다. Drive 저장을 켰다면 인증 화면을 직접 완료하세요.


In [ ]:
#@title 2. 환경·가중치 준비
HELPER_SOURCES = {'setup_environment.py': '"""무료 Colab의 기본 Python과 분리하여 SSPNet 실행 환경을 만든다."""\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nROOT = Path("/content/sspnet_free")\nREPO = ROOT / "SSPNet"\nVENV = ROOT / "venv"\nPYTHON = VENV / "bin/python"\nCOMMIT = "f2d0f42273945577a5c6497fd2b384cb545f8f66"\nSTAMP = "python38-torch110-cu111-mmcv1318-v1"\n\n\ndef command(*args, env=None):\n    subprocess.run([str(a) for a in args], check=True, env=env)\n\n\ndef setup():\n    if platform.system() != "Linux" or platform.machine() != "x86_64" or not Path("/content").is_dir():\n        raise RuntimeError("이 설치 셀은 Google Colab Linux 런타임에서 실행하세요.")\n    if not shutil.which("nvidia-smi"):\n        raise RuntimeError("무료 Colab에서 런타임 유형 변경 → GPU를 선택한 뒤 다시 실행하세요.")\n    command("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")\n    free = shutil.disk_usage("/content").free / 1024**3\n    print("Colab 임시 디스크 여유: %.1f GiB" % free, flush=True)\n    if free < 8:\n        raise RuntimeError("환경 설치를 시작하려면 임시 디스크에 최소 8 GiB 여유가 필요합니다.")\n    ROOT.mkdir(exist_ok=True)\n    if not PYTHON.is_file():\n        command(sys.executable,"-m","pip","install","--quiet","--no-cache-dir","uv==0.9.7")\n        command("uv","python","install","3.8.20")\n        command("uv","venv","--python","3.8.20","--seed",VENV)\n    env = os.environ.copy()\n    env.pop("PYTHONPATH",None)\n    env.pop("PYTHONHOME",None)\n    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"\n    env["PYTHONNOUSERSITE"] = "1"\n    marker = ROOT / "environment_ready.json"\n    ready = json.loads(marker.read_text()).get("stamp") == STAMP if marker.exists() else False\n    if not ready:\n        command(PYTHON,"-m","pip","install","--no-cache-dir","pip==24.3.1",\n                "setuptools==59.5.0","wheel==0.45.1",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "torch==1.10.0+cu111","torchvision==0.11.1+cu111",\n                "--find-links","https://download.pytorch.org/whl/cu111/torch_stable.html",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "numpy==1.21.6","scipy==1.7.3","matplotlib==3.5.3","Pillow==9.5.0",\n                "opencv-python-headless==4.5.5.64","pycocotools==2.0.7",\n                "addict==2.4.0","yapf==0.31.0","PyYAML==6.0.1","packaging==23.2",\n                "terminaltables==3.1.10","six==1.16.0","tqdm==4.66.5","gdown==5.2.0",env=env)\n        # 사전 빌드된 연산자만 설치한다. 무료 GPU 시간으로 CUDA 소스 빌드를 하지 않는다.\n        command(PYTHON,"-m","pip","install","--no-cache-dir","--no-deps",\n                "https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/"\n                "mmcv_full-1.3.18-cp38-cp38-manylinux1_x86_64.whl",env=env)\n    if not REPO.exists():\n        command("git","init",REPO)\n        command("git","-C",REPO,"remote","add","origin","https://github.com/MingboHong/SSPNet.git")\n        command("git","-C",REPO,"fetch","--depth=1","origin",COMMIT)\n        command("git","-C",REPO,"checkout","--detach","FETCH_HEAD")\n    actual = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()\n    if actual != COMMIT:\n        raise RuntimeError("기존 SSPNet 코드 버전이 다릅니다. 새 무료 런타임에서 설치 셀부터 실행하세요.")\n    env["PYTHONPATH"] = str(REPO)\n    check = \'\'\'\nimport json, sys, torch, torchvision, mmcv, mmdet\nfrom mmcv.ops import nms, roi_align\nassert torch.__version__ == \'1.10.0+cu111\'\nassert mmcv.__version__ == \'1.3.18\'\nassert torch.cuda.is_available(), \'GPU를 사용할 수 없습니다.\'\na = torch.randn(32,32,device=\'cuda\'); (a @ a).sum().item()\nb = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]],device=\'cuda\')\ns = torch.tensor([.9,.8],device=\'cuda\')\nnms(b,s,.5)\nroi_align(torch.ones(1,1,16,16,device=\'cuda\'),torch.tensor([[0.,1.,1.,8.,8.]],device=\'cuda\'),(3,3))\ntorch.cuda.synchronize()\nprint(json.dumps(dict(python=sys.version,torch=torch.__version__,torchvision=torchvision.__version__,\n    cuda=torch.version.cuda,mmcv=mmcv.__version__,mmdet=mmdet.__version__,\n    gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0)),ensure_ascii=False,indent=2))\n\'\'\'\n    command(PYTHON,"-c",check,env=env)\n    marker.write_text(json.dumps(dict(stamp=STAMP,source_commit=actual)),encoding="utf-8")\n    freeze = subprocess.check_output([str(PYTHON),"-m","pip","freeze"],env=env,text=True)\n    (ROOT/"installed_packages.txt").write_text(freeze,encoding="utf-8")\n    print("환경 확인 완료: CUDA 행렬 연산, NMS, RoIAlign, SSPNet 패키지 가져오기 성공",flush=True)\n\n\nif __name__ == "__main__":\n    setup()\n', 'metrics.py': '"""원본 이미지 좌표에서 수영자 탐지와 크기별 재현율을 평가한다."""\nimport json\nimport math\nimport re\nimport os\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef canonical(value):\n    return " ".join(re.sub(r"[_-]+", " ", value.lower()).split())\n\n\ndef image_path(annotation_path, name):\n    base = Path(annotation_path).resolve().parent\n    path = (base / name).resolve()\n    if base != path and base not in path.parents:\n        raise ValueError("이미지 경로가 데이터셋 폴더 밖을 가리킵니다: " + name)\n    if not path.is_file():\n        raise FileNotFoundError("정답 파일과 같은 폴더 아래에 이미지가 필요합니다: " + str(path))\n    return path\n\n\ndef size_group(box):\n    d = max(float(box[2]), float(box[3]))\n    return "0<d<=8" if d <= 8 else "8<d<=12" if d <= 12 else "12<d<=20" if d <= 20 else "20<d"\n\n\ndef geometric_size(box):\n    return math.sqrt(float(box[2]) * float(box[3]))\n\n\ndef merge_splits(annotation_paths, output_path):\n    """분할별로 겹치는 ID와 파일명을 구분하고 이미지 복사 없이 합친다."""\n    output_path = Path(output_path).resolve()\n    base = output_path.parent\n    merged = dict(info={"description": "전체 분할 통합", "source_splits": []},\n                  licenses=[], images=[], annotations=[], categories=[])\n    categories = {}\n    seen_paths = set()\n    for split, source in annotation_paths:\n        source = Path(source).resolve()\n        if split in merged["info"]["source_splits"]:\n            raise ValueError("같은 분할을 두 번 합칠 수 없습니다: " + split)\n        raw = json.loads(source.read_text(encoding="utf-8"))\n        merged["info"]["source_splits"].append(split)\n        catmap = {}\n        for category in raw["categories"]:\n            name = canonical(category["name"])\n            if name not in categories:\n                categories[name] = len(categories) + 1\n                merged["categories"].append(dict(id=categories[name], name=category["name"]))\n            catmap[category["id"]] = categories[name]\n        images = {}\n        for im in sorted(raw["images"], key=lambda v: v["id"]):\n            if im["id"] in images:\n                raise ValueError("분할 안에서 이미지 ID가 중복됩니다: " + split)\n            path = image_path(source, im["file_name"])\n            if path in seen_paths:\n                raise ValueError("같은 이미지 경로가 중복됩니다: " + str(path))\n            seen_paths.add(path)\n            if base not in path.parents:\n                raise ValueError("통합 정답은 모든 이미지의 상위 폴더에 저장해야 합니다.")\n            new_id = len(merged["images"])\n            images[im["id"]] = new_id\n            merged["images"].append(dict(im, id=new_id, file_name=path.relative_to(base).as_posix(),\n                                         source_split=split, source_image_id=im["id"]))\n        seen_annotations = set()\n        for a in raw["annotations"]:\n            if a["id"] in seen_annotations:\n                raise ValueError("분할 안에서 정답 ID가 중복됩니다: " + split)\n            seen_annotations.add(a["id"])\n            if a["image_id"] not in images or a["category_id"] not in catmap:\n                raise ValueError("정답의 이미지 또는 종류 ID를 찾지 못했습니다: " + split)\n            merged["annotations"].append(dict(a, id=len(merged["annotations"])+1,\n                image_id=images[a["image_id"]], category_id=catmap[a["category_id"]],\n                source_split=split, source_annotation_id=a["id"]))\n    if not merged["images"]:\n        raise ValueError("합칠 이미지가 없습니다.")\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(str(temporary), str(output_path))\n    return output_path\n\n\ndef load_dataset(annotation_path, target_names=("person water",), limit=0):\n    raw = json.loads(Path(annotation_path).read_text(encoding="utf-8"))\n    cats = raw.get("categories", [])\n    wanted = {canonical(s) for s in target_names}\n    selected = [c["id"] for c in cats if canonical(c["name"]) in wanted]\n    if not selected:\n        raise ValueError("물속 사람 라벨을 찾지 못했습니다. 실제 종류: " + str([c["name"] for c in cats]))\n    images = sorted(raw["images"], key=lambda im: im["id"])\n    if len({im["id"] for im in images}) != len(images):\n        raise ValueError("이미지 ID가 중복되어 있습니다.")\n    if limit < 0:\n        raise ValueError("이미지 제한은 0 이상이어야 합니다.")\n    if limit:\n        images = images[:limit]\n    if not images:\n        raise ValueError("선택한 분할에 이미지가 없습니다.")\n    imap = {im["id"]: im for im in images}\n    ann_ids = set()\n    annotations = []\n    for a in raw["annotations"]:\n        if a["id"] in ann_ids:\n            raise ValueError("정답 ID가 중복되어 있습니다.")\n        ann_ids.add(a["id"])\n        if a["image_id"] not in imap or a["category_id"] not in selected:\n            continue\n        x, y, w, h = map(float, a["bbox"])\n        im = imap[a["image_id"]]\n        if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:\n            raise ValueError("유효하지 않은 정답 박스: " + str(a["id"]))\n        if x < -.5 or y < -.5 or x + w > im["width"] + .5 or y + h > im["height"] + .5:\n            raise ValueError("이미지 밖의 정답 박스입니다. 좌표 변환을 확인하세요: " + str(a["id"]))\n        annotations.append(dict(id=a["id"], image_id=a["image_id"], category_id=1,\n                                bbox=[x, y, w, h], area=w*h,\n                                iscrowd=int(a.get("iscrowd", 0)),\n                                ignore=bool(a.get("ignore", False)), segmentation=[]))\n    for im in images:\n        path = image_path(annotation_path, im["file_name"])\n        from PIL import Image\n        with Image.open(path) as pic:\n            if pic.size != (im["width"], im["height"]):\n                raise ValueError("이미지 크기와 정답 메타데이터가 다릅니다: " + str(path))\n    gt = dict(info={"description": "물속 사람 평가용 정답"}, licenses=[], images=images,\n              annotations=annotations, categories=[dict(id=1, name="person water")])\n    counts = Counter(size_group(a["bbox"]) for a in annotations if not a["ignore"] and not a["iscrowd"])\n    normal = [a for a in annotations if not a["ignore"] and not a["iscrowd"]]\n    split_images = Counter(im.get("source_split", "single") for im in images)\n    image_splits = {im["id"]: im.get("source_split", "single") for im in images}\n    split_targets = Counter(image_splits[a["image_id"]] for a in normal)\n    return gt, dict(image_count=len(images), available_image_count=len(raw["images"]), target_count=sum(counts.values()),\n                    geometric_2to20_count=sum(2 <= geometric_size(a["bbox"]) < 20 for a in normal),\n                    geometric_2to8_count=sum(2 <= geometric_size(a["bbox"]) < 8 for a in normal),\n                    split_counts={s: dict(images=n, water_targets=split_targets[s]) for s,n in split_images.items()},\n                    size_counts=dict(counts), available_classes=[c["name"] for c in cats],\n                    target_classes=[c["name"] for c in cats if c["id"] in selected])\n\n\ndef overlap(a, b, crowd=False):\n    x, y, w, h = a\n    u, v, s, t = b\n    inter = max(0., min(x+w, u+s)-max(x, u)) * max(0., min(y+h, v+t)-max(y, v))\n    area = w*h if crowd else w*h+s*t-inter\n    return inter / area if area > 0 else 0.\n\n\ndef match_predictions(gt, predictions, iou=.5):\n    normal, ignored = defaultdict(list), defaultdict(list)\n    for a in gt["annotations"]:\n        (ignored if a.get("ignore") or a.get("iscrowd") else normal)[a["image_id"]].append(a)\n    used = set()\n    valid_ids = {im["id"] for im in gt["images"]}\n    ranked = sorted((p for p in predictions if p["image_id"] in valid_ids), key=lambda p: -p["score"])\n    matches = []\n    for p in ranked:\n        candidates = [(overlap(p["bbox"], a["bbox"]), a) for a in normal[p["image_id"]] if a["id"] not in used]\n        score, best = max(candidates, key=lambda pair: pair[0]) if candidates else (0., None)\n        if best is not None and score >= iou:\n            used.add(best["id"])\n            matches.append((p, best, False))\n        elif any(overlap(p["bbox"], a["bbox"], bool(a.get("iscrowd"))) >= iou for a in ignored[p["image_id"]]):\n            matches.append((p, None, True))\n        else:\n            matches.append((p, None, False))\n    return matches\n\n\ndef evaluate(gt, predictions, score_threshold=.25, iou=.5):\n    matches = match_predictions(gt, predictions, iou)\n    counts = Counter(size_group(a["bbox"]) for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd"))\n    positives = sum(counts.values())\n    hits, tp, fp = Counter(), 0, 0\n    recalls, precisions = [], []\n    total_tp = total_fp = 0\n    for p, a, ignored in matches:\n        if ignored:\n            continue\n        total_tp += a is not None\n        total_fp += a is None\n        recalls.append(total_tp / positives if positives else 0.)\n        precisions.append(total_tp / (total_tp + total_fp))\n        if p["score"] >= score_threshold:\n            tp += a is not None\n            fp += a is None\n            if a is not None:\n                hits[size_group(a["bbox"])] += 1\n    ap = sum(max((p for r, p in zip(recalls, precisions) if r >= q/100), default=0.) for q in range(101))/101 if positives else None\n    tiny_count = sum(counts[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    tiny_hits = sum(hits[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    def geometric_range(low, high):\n        selected = {a["id"] for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd")\n                    and low <= geometric_size(a["bbox"]) < high}\n        found = sum(a is not None and a["id"] in selected for p,a,ignored in matches\n                    if not ignored and p["score"] >= score_threshold)\n        return dict(positives=len(selected), hits=found, recall=found/len(selected) if selected else None)\n    return dict(iou=iou, score_threshold=score_threshold, images=len(gt["images"]),\n                positives=positives, true_positives=tp, false_positives=fp,\n                precision=tp/(tp+fp) if tp+fp else None,\n                recall=tp/positives if positives else None,\n                false_positives_per_image=fp/len(gt["images"]),\n                AP50_saved_predictions_101point=ap if iou == .5 else None,\n                geometric_size_definition="s=sqrt(width*height), 배포 이미지 픽셀 기준",\n                geometric_2to20=geometric_range(2,20),\n                geometric_2to8=geometric_range(2,8),\n                geometric_size_recall={"2<=s<8": geometric_range(2,8),\n                                       "8<=s<12": geometric_range(8,12),\n                                       "12<=s<20": geometric_range(12,20)},\n                tiny_le20=dict(positives=tiny_count, hits=tiny_hits,\n                               recall=tiny_hits/tiny_count if tiny_count else None),\n                size_recall={k: dict(positives=counts[k], hits=hits[k],\n                                    recall=hits[k]/counts[k] if counts[k] else None)\n                             for k in ("0<d<=8", "8<d<=12", "12<d<=20", "20<d")})\n', 'engine.py': '"""SSPNet 추론, 분할 병합, 이미지별 재시작 캐시."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nfrom metrics import evaluate, image_path, load_dataset\n\n\ndef dump(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(str(tmp), str(path))\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024*1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef positions(length, size, overlap):\n    if size <= 0 or not 0 <= overlap < 1:\n        raise ValueError("분할 크기는 양수, 겹침은 0 이상 1 미만이어야 합니다.")\n    if length <= size:\n        return [0]\n    step = max(1, int(size*(1-overlap)))\n    values = list(range(0, length-size+1, step))\n    if values[-1] != length-size:\n        values.append(length-size)\n    return values\n\n\ndef merge_nms(boxes, threshold=.5):\n    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 5)\n    if not len(boxes):\n        return boxes\n    order = np.argsort(-boxes[:, 4], kind="stable")\n    keep = []\n    while len(order):\n        i = order[0]\n        keep.append(i)\n        rest = order[1:]\n        b = boxes[rest]\n        inter = np.maximum(0, np.minimum(boxes[i, 2:4], b[:, 2:4])-np.maximum(boxes[i, :2], b[:, :2])).prod(axis=1)\n        area = np.maximum(0, boxes[i, 2:4]-boxes[i, :2]).prod()\n        areas = np.maximum(0, b[:, 2:4]-b[:, :2]).prod(axis=1)\n        union = area+areas-inter\n        iou = inter/np.maximum(union, 1e-12)\n        order = rest[iou <= threshold]\n    return boxes[keep]\n\n\ndef make_model(settings):\n    import torch\n    from mmcv import Config\n    from mmdet.models import build_detector\n    if not torch.cuda.is_available():\n        raise RuntimeError("GPU가 없습니다. 무료 Colab에서 GPU 런타임을 선택하세요.")\n    torch.set_num_threads(2)\n    torch.manual_seed(0)\n    np.random.seed(0)\n    torch.backends.cudnn.benchmark = False\n    cfg = Config.fromfile(str(Path(settings["repo"])/"configs/_base_/models/faster_rcnn_r50_sspnet.py"))\n    cfg.model.pretrained = None\n    cfg.model.train_cfg = None\n    cfg.test_cfg.rcnn.score_thr = settings["save_min_score"]\n    cfg.test_cfg.rcnn.nms.min_score = settings["save_min_score"]\n    cfg.test_cfg.rcnn.max_per_img = settings["max_detections"]\n    cfg.data = dict(test=dict(pipeline=[\n        dict(type="LoadImageFromFile"),\n        dict(type="MultiScaleFlipAug", img_scale=(settings["long_edge"], settings["short_edge"]),\n             flip=False, transforms=[dict(type="Resize", keep_ratio=True), dict(type="RandomFlip"),\n             dict(type="Normalize", mean=[123.675,116.28,103.53], std=[58.395,57.12,57.375], to_rgb=True),\n             dict(type="Pad", size_divisor=32), dict(type="ImageToTensor", keys=["img"]),\n             dict(type="Collect", keys=["img"])])]))\n    model = build_detector(cfg.model, test_cfg=cfg.test_cfg)\n    checkpoint = torch.load(settings["checkpoint"], map_location="cpu")\n    state = checkpoint.get("state_dict", checkpoint)\n    if all(k.startswith("module.") for k in state):\n        state = {k[7:]: v for k,v in state.items()}\n    if not any("neck.CAM." in k for k in state):\n        raise ValueError("SSPNet 가중치가 아닙니다. Faster R-CNN SSPNet용 파일을 확인하세요.")\n    model.load_state_dict(state, strict=True)\n    model.CLASSES = ("person",)\n    model.cfg = cfg\n    model.cuda().eval()\n    return model, cfg\n\n\ndef predict(model, image, settings):\n    import torch\n    from mmdet.apis import inference_detector\n    rgb = np.asarray(Image.open(image).convert("RGB"))\n    h, w = rgb.shape[:2]\n    torch.cuda.reset_peak_memory_stats()\n    torch.cuda.synchronize()\n    started = time.perf_counter()\n    patches = 0\n    boxes = []\n    if settings["mode"] == "full":\n        regions = [(0, 0, w, h)]\n    else:\n        size = settings["tile_size"]\n        regions = [(x,y,min(x+size,w),min(y+size,h))\n                   for y in positions(h,size,settings["tile_overlap"])\n                   for x in positions(w,size,settings["tile_overlap"])]\n    for x,y,r,b in regions:\n        # MMDetection의 배열 입력은 BGR 순서를 사용한다.\n        patch = np.ascontiguousarray(rgb[y:b,x:r,::-1])\n        result = inference_detector(model, patch)\n        if not isinstance(result, list) or len(result) != 1:\n            raise ValueError("단일 person 클래스 SSPNet 출력이 아닙니다.")\n        out = np.asarray(result[0], dtype=np.float32).reshape(-1,5).copy()\n        out[:,[0,2]] += x\n        out[:,[1,3]] += y\n        boxes.extend(out.tolist())\n        patches += 1\n    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1,5)\n    if len(boxes):\n        boxes[:,[0,2]] = boxes[:,[0,2]].clip(0,w)\n        boxes[:,[1,3]] = boxes[:,[1,3]].clip(0,h)\n        valid = np.isfinite(boxes).all(axis=1) & (boxes[:,2]>boxes[:,0]) & (boxes[:,3]>boxes[:,1])\n        boxes = boxes[valid]\n    if settings["mode"] == "tiles":\n        boxes = merge_nms(boxes,settings["merge_iou"])\n    if len(boxes):\n        boxes = boxes[np.argsort(-boxes[:,4],kind="stable")][:settings["max_detections"]]\n    torch.cuda.synchronize()\n    return dict(boxes=boxes.tolist(), seconds=time.perf_counter()-started, patches=patches,\n                peak_vram_mib=torch.cuda.max_memory_allocated()/1024**2)\n\n\ndef draw(image, record, path, threshold):\n    picture = Image.open(image).convert("RGB")\n    pen = ImageDraw.Draw(picture)\n    for x,y,r,b,s in record["boxes"]:\n        if s >= threshold:\n            pen.rectangle((x,y,r,b),outline="red",width=2)\n            pen.text((x,max(0,y-12)),"person %.2f" % s,fill="red")\n    picture.save(path)\n\n\ndef coco_predictions(records):\n    return [dict(image_id=im_id,category_id=1,bbox=[x,y,r-x,b-y],score=s)\n            for im_id,rec in records.items() for x,y,r,b,s in rec["boxes"]]\n\n\ndef report_text(metrics, settings, summary):\n    def percent(value):\n        return "계산 불가" if value is None else "%.1f%%" % (100*value)\n    small = metrics["geometric_2to20"]\n    smallest = metrics["geometric_2to8"]\n    method = "이미지 전체를 한 번에 검사" if settings["mode"] == "full" else \\\n        "%d픽셀 패치로 분할 검사(겹침 %.0f%%)" % (settings["tile_size"],settings["tile_overlap"]*100)\n    lines = ["SSPNet 수영자 탐지 평가", "",\n             "평가 데이터: %s / 분할: %s / 이미지 %d장" %\n             (settings.get("dataset_name","사용자 데이터"), settings.get("split","single"), metrics["images"]),\n             "평가 방식: " + method,\n             "공개 TinyPerson 학습 가중치를 사용했으며 이번 데이터로 추가 학습하지 않았다.",\n             "전체 물속 사람 정답 %d개 중 %d개를 찾아 재현율 %s, 정밀도 %s였다." %\n             (metrics["positives"],metrics["true_positives"],percent(metrics["recall"]),percent(metrics["precision"])),\n             "사람 크기는 배포 이미지에서 s=√(박스 가로×박스 세로)로 정의했다.",\n             "2≤s<20픽셀 물속 사람 %d개 중 %d개를 찾아 재현율 %s였다." %\n             (small["positives"],small["hits"],percent(small["recall"])),\n             "가장 작은 2≤s<8픽셀 구간에서는 %d개 중 %d개를 찾아 재현율 %s였다." %\n             (smallest["positives"],smallest["hits"],percent(smallest["recall"])),\n             "오탐은 %d개(이미지당 %.2f개)였고, 점수 임계값 %.3f와 IoU %.2f를 적용했다." %\n             (metrics["false_positives"],metrics["false_positives_per_image"],metrics["score_threshold"],metrics["iou"])]\n    if settings.get("split") == "all":\n        lines.append("train·valid·test와 배포된 증강 이미지를 포함한 전체 분할 평가이며, 독립 테스트 분할 성능과 구분한다.")\n    if metrics["partial"]:\n        lines.append("현재 결과는 일부 이미지만 처리한 표본 평가다.")\n    lines += ["", "분할별 구성: " + json.dumps(summary["split_counts"],ensure_ascii=False),\n              "입력 해상도 상한: %d×%d, 종횡비 유지 / 최대 탐지 수: %d" %\n              (settings["long_edge"],settings["short_edge"],settings["max_detections"]),\n              "검출 결과는 전체 물속 사람 정답과 일대일로 매칭한 뒤 크기 구간별로 집계했다.",\n              "이 파일은 이번 SSPNet 실행 결과만 정리하며 다른 모델과의 비교는 포함하지 않는다."]\n    return "\\n".join(lines) + "\\n"\n\n\ndef run(args):\n    import torch\n    settings = json.loads(Path(args.settings).read_text(encoding="utf-8"))\n    if settings["mode"] not in ("full","tiles"):\n        raise ValueError("추론 방식은 full 또는 tiles여야 합니다.")\n    if not 0 < settings["save_min_score"] <= settings["report_score"] <= 1:\n        raise ValueError("저장 점수와 보고 점수의 범위를 확인하세요.")\n    if not 0 < settings.get("match_iou",.5) <= 1:\n        raise ValueError("정답 매칭 IoU는 0 초과 1 이하여야 합니다.")\n    if settings["mode"] == "tiles":\n        positions(1,settings["tile_size"],settings["tile_overlap"])\n    out = Path(args.out)\n    out.mkdir(parents=True,exist_ok=True)\n    settings["checkpoint_sha256"] = sha256(settings["checkpoint"])\n    settings["source_commit"] = subprocess.check_output(["git","-C",settings["repo"],"rev-parse","HEAD"],text=True).strip()\n    settings["engine_sha256"] = sha256(__file__)\n    settings["metrics_sha256"] = sha256(Path(__file__).with_name("metrics.py"))\n    if args.image:\n        model,cfg = make_model(settings)\n        cfg.dump(str(out/"resolved_config.py"))\n        # 첫 호출의 초기화 비용은 별도 기록하고 본 측정에서 제외한다.\n        warmup = predict(model,args.image,dict(settings,mode="full"))\n        rec = predict(model,args.image,settings)\n        rec.update(image_sha256=sha256(args.image), image_file_name=Path(args.image).name)\n        dump(out/"predictions.json",rec)\n        dump(out/"settings.json",dict(settings,warmup_seconds=warmup["seconds"],gpu=torch.cuda.get_device_name(0)))\n        draw(args.image,rec,out/"prediction.jpg",settings["report_score"])\n        print("이미지 추론 완료: 박스 %d개, 표시 임계값 %.2f, %.3f초" %\n              (len(rec["boxes"]),settings["report_score"],rec["seconds"]))\n        print("출력:",out)\n        return\n    gt,summary = load_dataset(args.annotations,settings["target_names"],args.limit)\n    if not summary["target_count"]:\n        raise ValueError("선택한 이미지에 평가할 물속 사람 정답이 없습니다.")\n    dump(out/"dataset_summary.json",summary)\n    settings["annotations_sha256"] = sha256(args.annotations)\n    identity = dict(settings)\n    # 결과 표시 임계값을 바꿔도 저점수 예측을 재사용할 수 있다.\n    identity.pop("report_score",None)\n    key = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:16]\n    cache = out/("cache_"+key)\n    cache.mkdir(exist_ok=True)\n    dump(cache/"identity.json",identity)\n    dump(out/"run_status.json",dict(state="실행 중",cache_key=key,\n         image_ids=[im["id"] for im in gt["images"]]))\n    records = {}\n    pending = []\n    for im in gt["images"]:\n        path = image_path(args.annotations,im["file_name"])\n        image_hash = sha256(path)\n        saved = cache/(str(im["id"])+".json")\n        try:\n            rec = json.loads(saved.read_text(encoding="utf-8"))\n        except (FileNotFoundError,json.JSONDecodeError):\n            rec = None\n        if rec and rec.get("image_sha256") == image_hash:\n            records[im["id"]] = rec\n        else:\n            pending.append((im,path,image_hash,saved))\n    warmup_seconds = None\n    if pending:\n        model,cfg = make_model(settings)\n        cfg.dump(str(out/"resolved_config.py"))\n        warmup_seconds = predict(model,pending[0][1],dict(settings,mode="full"))["seconds"]\n        for im,path,image_hash,saved in pending:\n            rec = predict(model,path,settings)\n            rec.update(image_sha256=image_hash,file_name=im["file_name"])\n            dump(saved,rec)\n            records[im["id"]] = rec\n            print("완료 %d/%d: %s" % (len(records),len(gt["images"]),im["file_name"]),flush=True)\n            # 적은 수의 시각화만 저장하여 무료 저장공간을 아낀다.\n            if len(records) <= 5:\n                draw(path,rec,out/("preview_%s.jpg" % im["id"]),settings["report_score"])\n    predictions = coco_predictions(records)\n    dump(out/"ground_truth_water.json",gt)\n    dump(out/"predictions_coco.json",predictions)\n    metrics = evaluate(gt,predictions,settings["report_score"],settings.get("match_iou",.5))\n    metrics.update(partial=len(gt["images"]) < summary["available_image_count"],requested_limit=args.limit,cache_key=key,\n                   mean_inference_seconds=sum(r["seconds"] for r in records.values())/len(records),\n                   maximum_vram_mib=max(r["peak_vram_mib"] for r in records.values()),\n                   warmup_seconds=warmup_seconds)\n    metrics["by_split"] = {}\n    for split in summary["split_counts"]:\n        image_ids = {im["id"] for im in gt["images"] if im.get("source_split","single") == split}\n        split_gt = dict(gt, images=[im for im in gt["images"] if im["id"] in image_ids],\n                        annotations=[a for a in gt["annotations"] if a["image_id"] in image_ids])\n        metrics["by_split"][split] = evaluate(split_gt,predictions,settings["report_score"],settings.get("match_iou",.5))\n    dump(out/"metrics.json",metrics)\n    (out/"문서용_결과요약.txt").write_text(report_text(metrics,settings,summary),encoding="utf-8")\n    dump(out/"run_manifest.json",dict(settings,dataset=summary,\n         image_ids=[im["id"] for im in gt["images"]],gpu=torch.cuda.get_device_name(0)))\n    dump(out/"run_status.json",dict(state="완료",cache_key=key,\n         image_ids=[im["id"] for im in gt["images"]]))\n    with open(out/"metrics.csv","w",encoding="utf-8-sig",newline="") as f:\n        writer=csv.writer(f)\n        writer.writerow(["지표","값"])\n        for name,value in metrics.items():\n            if not isinstance(value,dict):\n                writer.writerow([name,value])\n        for name,value in metrics["size_recall"].items():\n            writer.writerow(["크기별 재현율 "+name,value["recall"]])\n        for name in ("geometric_2to20","geometric_2to8"):\n            for item,value in metrics[name].items():\n                writer.writerow([name+" "+item,value])\n        for name,value in metrics["geometric_size_recall"].items():\n            writer.writerow(["기하평균 크기별 재현율 "+name,value["recall"]])\n    print(json.dumps(metrics,ensure_ascii=False,indent=2))\n\n\nif __name__ == "__main__":\n    parser=argparse.ArgumentParser(description="SSPNet 단일 이미지 추론과 물속 사람 평가")\n    parser.add_argument("--settings",required=True)\n    parser.add_argument("--out",required=True)\n    group=parser.add_mutually_exclusive_group(required=True)\n    group.add_argument("--image")\n    group.add_argument("--annotations")\n    parser.add_argument("--limit",type=int,default=0)\n    run(parser.parse_args())\n', 'summarize_scopes.py': '"""한 번 저장한 person 예측을 전체 사람·물속 사람 기준으로 각각 집계한다."""\nimport argparse\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom metrics import geometric_size, load_dataset, match_predictions\n\n\ndef score_scope(gt, predictions, score, iou):\n    selected = [p for p in predictions if p["score"] >= score]\n    matches = match_predictions(gt, selected, iou)\n    normal = [a for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd")]\n    annotations_by_image = defaultdict(list)\n    matches_by_image = defaultdict(list)\n    for annotation in normal:\n        annotations_by_image[annotation["image_id"]].append(annotation)\n    for record in matches:\n        matches_by_image[record[0]["image_id"]].append(record)\n\n    def group(images):\n        annotations = [a for im in images for a in annotations_by_image[im["id"]]]\n        records = [r for im in images for r in matches_by_image[im["id"]]]\n        hits = [a for _, a, ignored in records if a is not None and not ignored]\n        false_positive = sum(a is None and not ignored for _, a, ignored in records)\n        true_positive = len(hits)\n        def sizes(low, high):\n            positives = sum(low <= geometric_size(a["bbox"]) < high for a in annotations)\n            found = sum(low <= geometric_size(a["bbox"]) < high for a in hits)\n            return dict(positives=positives, hits=found, recall=found/positives if positives else None)\n        return dict(images=len(images), positives=len(annotations), true_positives=true_positive,\n                    false_positives=false_positive, false_negatives=len(annotations)-true_positive,\n                    precision=true_positive/(true_positive+false_positive) if true_positive+false_positive else None,\n                    recall=true_positive/len(annotations) if annotations else None,\n                    false_positives_per_image=false_positive/len(images),\n                    geometric_2to20=sizes(2,20), geometric_2to8=sizes(2,8))\n\n    result = group(gt["images"])\n    result.update(score_threshold=score, iou=iou,\n                  geometric_size_definition="s=sqrt(width*height), 배포 이미지 픽셀 기준")\n    splits = sorted({im.get("source_split", "single") for im in gt["images"]})\n    result["by_split"] = {s:group([im for im in gt["images"] if im.get("source_split", "single")==s]) for s in splits}\n    return result\n\n\ndef summarize_scopes(annotations, out, score=.25, iou=.3):\n    if not 0 < score <= 1 or not 0 < iou <= 1:\n        raise ValueError("점수와 IoU 기준은 0 초과 1 이하여야 합니다.")\n    out = Path(out)\n    predictions = json.loads((out/"predictions_coco.json").read_text(encoding="utf-8"))\n    saved_path = out/"ground_truth_water.json"\n    image_ids = {im["id"] for im in json.loads(saved_path.read_text(encoding="utf-8"))["images"]} if saved_path.is_file() else None\n    scopes = (("all_people", ["person water","person sand"], 8099, "전체 사람"),\n              ("water_only", ["person water"], 2904, "물속 사람"))\n    values = {}\n    reference = {}\n    thresholds = [iou] if iou==.5 else [iou,.5]\n    for name, classes, expected, label in scopes:\n        gt, _ = load_dataset(annotations, classes)\n        available_images = len(gt["images"])\n        if image_ids is not None:\n            gt["images"] = [im for im in gt["images"] if im["id"] in image_ids]\n            gt["annotations"] = [a for a in gt["annotations"] if a["image_id"] in image_ids]\n        if not gt["images"]:\n            raise ValueError("평가할 이미지가 없습니다.")\n        gt["info"]["description"] = label+" 평가용 정답"\n        gt["categories"] = [dict(id=1,name="person" if name=="all_people" else "person water")]\n        if name=="all_people":\n            (out/"ground_truth_all_people.json").write_text(json.dumps(gt,ensure_ascii=False),encoding="utf-8")\n        for threshold in thresholds:\n            result = score_scope(gt, predictions, score, threshold)\n            result.update(target_classes=classes,scope_label=label,\n                          partial=len(gt["images"])<available_images,available_image_count=available_images)\n            primary = threshold==iou\n            filename = "metrics_%s%s.json" % (name,"" if primary else "_iou05_reference")\n            (out/filename).write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")\n            (values if primary else reference)[name] = result\n\n    def percent(value):\n        return "계산 불가" if value is None else "%.1f%%" % (100*value)\n    lines = ["SSPNet 전체 사람·물속 사람 동시 평가", "", "동일한 person 예측을 두 가지 정답 범위로 각각 평가했습니다.",\n             "이미지: %d장 / 예측 점수: %.2f / 주 결과 IoU: %.2f" % (values["all_people"]["images"],score,iou), ""]\n    for name, _, _, label in scopes:\n        m = values[name]\n        lines += ["[%s 기준]" % label,\n                  "정답 %s개 / 정탐 %s개 / 놓친 정답 %s개" % tuple(format(m[k],",") for k in ("positives","true_positives","false_negatives")),\n                  "재현율 %s / 정밀도 %s" % (percent(m["recall"]),percent(m["precision"])),\n                  "주석 기준 오탐 %s개 / 이미지당 %.2f개" % (format(m["false_positives"],","),m["false_positives_per_image"]), ""]\n    lines += ["전체 사람 기준에서는 모래사장 사람도 정답입니다.",\n              "물속 사람 기준에서는 모래사장 사람에 연결되는 예측도 오탐에 포함됩니다.",\n              "주석 누락·위치 오류는 보정하지 않았습니다. 동일한 예측을 범위별로 다시 매칭하므로 정탐 수는 단순 합산으로 연결되지 않을 수 있습니다.",\n              "크기 제한 없는 결과가 주 결과이며 크기별 재현율과 train·valid·test별 지표도 각 JSON에 저장했습니다.",\n              "IoU 0.5 참고 결과도 저장했습니다." if reference else "현재 IoU 0.5로 집계했습니다."]\n    report = "\\n".join(lines)+"\\n"\n    (out/"전체사람_물속사람_결과요약.txt").write_text(report,encoding="utf-8")\n    print(report,flush=True)\n    return values\n\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser(description="전체 사람과 물속 사람의 동시 결과 집계")\n    parser.add_argument("--annotations",required=True)\n    parser.add_argument("--out",required=True)\n    parser.add_argument("--score",type=float,default=.25)\n    parser.add_argument("--iou",type=float,default=.3)\n    args=parser.parse_args()\n    summarize_scopes(args.annotations,args.out,args.score,args.iou)\n', 'workflow.py': '"""Colab 설정·데이터 준비·실행·비교·다운로드를 위한 공통 처리."""\nimport csv\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport subprocess\nimport os\nimport zipfile\nfrom pathlib import Path\n\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef dump(path,value):\n    path=Path(path)\n    path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")\n\n\ndef prepare_dataset(config,work,zip_path):\n    from metrics import merge_splits,load_dataset\n    work=Path(work)\n    signature=digest(zip_path)\n    root=work/"datasets"/config["dataset_name"]/signature[:12]\n    stamp=root/"prepared.json"\n    reused=stamp.is_file() and json.loads(stamp.read_text()).get("zip_sha256")==signature\n    if not reused:\n        root.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(zip_path) as archive:\n            if sum(m.file_size for m in archive.infolist())+1024**3>shutil.disk_usage(work).free:\n                raise RuntimeError("데이터 압축 해제에 필요한 공간이 부족합니다.")\n            for member in archive.infolist():\n                resolved=(root/member.filename).resolve()\n                if root.resolve()!=resolved and root.resolve() not in resolved.parents:\n                    raise ValueError("ZIP에 데이터 폴더 밖을 가리키는 경로가 있습니다.")\n                if (member.external_attr>>16)&0o170000==0o120000:\n                    raise ValueError("ZIP의 심볼릭 링크는 지원하지 않습니다.")\n            archive.extractall(root)\n        dump(stamp,dict(zip_sha256=signature))\n    split=config["split"]\n    parts=("train","valid","test") if split=="all" else (split,)\n    sources=[]\n    for part in parts:\n        paths=[p for p in root.rglob("_annotations.coco.json") if p.parent.name==part]\n        if len(paths)!=1:\n            raise ValueError(part+" 분할의 정답 JSON을 한 개 찾을 수 없습니다.")\n        sources.append((part,str(paths[0])))\n    annotations=root/("merged_%s.coco.json"%split)\n    merge_splits(sources,annotations)\n    gt,summary=load_dataset(annotations,["person water","person sand"])\n    water,water_summary=load_dataset(annotations,["person water"])\n    if config["dataset_name"]=="AnnotationPiriapolis_v4" and split=="all":\n        if (summary["image_count"],summary["target_count"],water_summary["target_count"])!=(635,8099,2904):\n            raise ValueError("AnnotationPiriapolis v4의 전체 635장·전체 사람 8099개·물속 사람 2904개와 다릅니다.")\n    state=dict(task="dataset",dataset_name=config["dataset_name"],split=split,\n               annotations=str(annotations),zip_sha256=signature,\n               images=summary["image_count"],all_people=summary["target_count"],water_only=water_summary["target_count"])\n    dump(work/"prepared_input.json",state)\n    print("데이터 준비 완료: %d장 / 전체 사람 %d개 / 물속 사람 %d개"%(state["images"],state["all_people"],state["water_only"]),flush=True)\n    print("기존 압축 해제 파일을 재사용했습니다." if reused else "데이터 압축을 해제했습니다.",flush=True)\n    return state\n\n\ndef import_baseline(zip_path,destination):\n    destination=Path(destination)\n    needed=("predictions_coco.json","ground_truth_water.json","run_manifest.json")\n    with zipfile.ZipFile(zip_path) as archive:\n        candidates=[n for n in archive.namelist() if n.endswith("/full/predictions_coco.json")]\n        if len(candidates)!=1:\n            raise ValueError("기존 전체 검사 결과 ZIP의 full/predictions_coco.json을 한 개 찾을 수 없습니다.")\n        prefix=candidates[0].rsplit("/",1)[0]+"/"\n        payload={name:archive.read(prefix+name) for name in needed}\n        manifest=json.loads(payload["run_manifest.json"])\n        if manifest["mode"]!="full":\n            raise ValueError("비교용 ZIP은 전체 검사(full) 결과여야 합니다.")\n    for name,data in payload.items():\n        json.loads(data)\n    destination.mkdir(parents=True,exist_ok=True)\n    for name,data in payload.items():\n        (destination/name).write_bytes(data)\n    return destination\n\n\ndef align_baseline(baseline,annotations,checkpoint,settings,destination):\n    from metrics import load_dataset\n    baseline,destination=Path(baseline),Path(destination)\n    manifest=json.loads((baseline/"run_manifest.json").read_text(encoding="utf-8"))\n    if manifest.get("checkpoint_sha256")!=digest(checkpoint):\n        raise ValueError("기존 전체 검사와 현재 가중치가 달라 같은 모델의 분할 효과를 비교할 수 없습니다.")\n    for key in ("long_edge","short_edge","max_detections","save_min_score"):\n        if manifest.get(key)!=settings.get(key):\n            raise ValueError("기존 전체 검사와 설정이 다릅니다: "+key)\n    gt,_=load_dataset(annotations,["person water"])\n    old=json.loads((baseline/"ground_truth_water.json").read_text(encoding="utf-8"))\n    def key(im):\n        return (im.get("source_split",""),im.get("source_image_id",im["id"]),Path(im["file_name"]).name)\n    old_map={key(im):im["id"] for im in old["images"]}\n    if any(key(im) not in old_map for im in gt["images"]):\n        raise ValueError("기존 전체 검사 결과에 현재 데이터의 이미지가 모두 포함돼 있지 않습니다.")\n    remap={old_map[key(im)]:im["id"] for im in gt["images"]}\n    # 이름이 같은 파일의 주석이 변경된 비교를 막는다.\n    def boxes(raw,ids=None):\n        values=[]\n        for a in raw["annotations"]:\n            image_id=a["image_id"]\n            if ids is not None:\n                if image_id not in ids:continue\n                image_id=ids[image_id]\n            values.append((image_id,tuple(a["bbox"]),bool(a.get("ignore")),bool(a.get("iscrowd"))))\n        return sorted(values)\n    if boxes(old,remap)!=boxes(gt):\n        raise ValueError("기존 전체 검사와 현재 물속 사람 주석이 달라 비교를 중단했습니다.")\n    predictions=json.loads((baseline/"predictions_coco.json").read_text(encoding="utf-8"))\n    predictions=[dict(p,image_id=remap[p["image_id"]]) for p in predictions if p["image_id"] in remap]\n    dump(destination/"predictions_coco.json",predictions)\n    dump(destination/"ground_truth_water.json",gt)\n    dump(destination/"run_manifest.json",dict(manifest,comparison_image_ids=list(remap.values())))\n    return destination\n\n\ndef cached_predictions_valid(out,settings,annotations,limit):\n    from metrics import load_dataset,image_path\n    out=Path(out)\n    try:\n        old=json.loads((out/"run_manifest.json").read_text(encoding="utf-8"))\n        status=json.loads((out/"run_status.json").read_text(encoding="utf-8"))\n        if status["state"]!="완료" or not (out/"predictions_coco.json").is_file():return False\n        keys=("mode","long_edge","short_edge","tile_size","tile_overlap","merge_iou","save_min_score","max_detections")\n        if any(old.get(k)!=settings.get(k) for k in keys):return False\n        if old["checkpoint_sha256"]!=digest(settings["checkpoint"]) or old["annotations_sha256"]!=digest(annotations):return False\n        if old.get("engine_sha256")!=digest(Path(settings["repo"]).parent/"helpers/engine.py"):return False\n        commit=subprocess.check_output(["git","-C",settings["repo"],"rev-parse","HEAD"],text=True).strip()\n        if old.get("source_commit")!=commit:return False\n        gt,_=load_dataset(annotations,["person water"],limit)\n        if old["image_ids"]!=[im["id"] for im in gt["images"]]:return False\n        cache=out/("cache_"+status["cache_key"])\n        return all(json.loads((cache/(str(im["id"])+".json")).read_text(encoding="utf-8"))["image_sha256"]==digest(image_path(annotations,im["file_name"])) for im in gt["images"])\n    except (OSError,ValueError,KeyError):\n        return False\n\n\ndef run_dataset(config,work,store,checkpoint,env):\n    from summarize_scopes import summarize_scopes\n    work,store=Path(work),Path(store)\n    state=json.loads((work/"prepared_input.json").read_text(encoding="utf-8"))\n    if any(state.get(k)!=config[k] for k in ("task","dataset_name","split")):\n        raise ValueError("데이터 선택이 변경됐습니다. 3단계부터 다시 실행하세요.")\n    mode=config["mode"]\n    label="full" if mode=="full" else "tiles_%d_overlap%g"%(config["tile_size"],config["tile_overlap"])\n    out=store/"evaluation"/config["dataset_name"]/config["split"]/label\n    if config.get("weight_source", "public") != "public":\n        out=store/"evaluation_models"/digest(checkpoint)[:16]/config["dataset_name"]/config["split"]/label\n    settings=dict(repo=str(work/"SSPNet"),checkpoint=str(checkpoint),mode=mode,\n                  long_edge=config["long_edge"],short_edge=config["short_edge"],tile_size=config["tile_size"],\n                  tile_overlap=config["tile_overlap"],merge_iou=.5,report_score=config["report_score"],\n                  save_min_score=.001,max_detections=500,target_names=["person water"],\n                  dataset_name=config["dataset_name"],split=config["split"],match_iou=config["match_iou"])\n    dump(work/"evaluation_settings.json",settings)\n    python=work/"venv/bin/python"\n    if cached_predictions_valid(out,settings,state["annotations"],config["max_images"]):\n        print("동일한 추론 결과를 재사용하고 점수만 다시 집계합니다.",flush=True)\n    else:\n        subprocess.run([str(python),str(work/"helpers/engine.py"),"--settings",str(work/"evaluation_settings.json"),\n                        "--annotations",state["annotations"],"--limit",str(config["max_images"]),"--out",str(out)],env=env,check=True)\n    summarize_scopes(state["annotations"],out,config["report_score"],config["match_iou"])\n    outputs=[out]\n    baseline=Path(config.get("baseline_path") or store/"baselines"/config["dataset_name"])\n    if config.get("compare_with_full") and mode!="full":\n        if not (baseline/"predictions_coco.json").is_file():\n            baseline=store/"evaluation"/config["dataset_name"]/config["split"]/"full"\n        if not (baseline/"predictions_coco.json").is_file():\n            raise ValueError("기존 전체 검사 결과가 없습니다. 3단계에서 비교용 ZIP을 준비하세요.")\n        destination=out/"baseline_full_rescored"\n        align_baseline(baseline,state["annotations"],checkpoint,settings,destination)\n        # 부분 평가에서는 양쪽 모두 현재 처리한 이미지로만 비교한다.\n        current=json.loads((out/"ground_truth_water.json").read_text(encoding="utf-8"))\n        aligned=json.loads((destination/"ground_truth_water.json").read_text(encoding="utf-8"))\n        ids={im["id"] for im in current["images"]}\n        aligned["images"]=[im for im in aligned["images"] if im["id"] in ids]\n        aligned["annotations"]=[a for a in aligned["annotations"] if a["image_id"] in ids]\n        dump(destination/"ground_truth_water.json",aligned)\n        summarize_scopes(state["annotations"],destination,config["report_score"],config["match_iou"])\n        rows=[]\n        for scope in ("all_people","water_only"):\n            for method,folder in (("전체 검사",destination),("분할 검사",out)):\n                m=json.loads((folder/("metrics_"+scope+".json")).read_text(encoding="utf-8"))\n                rows.append(dict(평가범위=m["scope_label"],방식=method,이미지=m["images"],정답=m["positives"],정탐=m["true_positives"],\n                                 재현율=m["recall"],정밀도=m["precision"],오탐=m["false_positives"],이미지당오탐=m["false_positives_per_image"],\n                                 작은사람재현율=m["geometric_2to20"]["recall"],최소구간재현율=m["geometric_2to8"]["recall"]))\n        dump(out/"전체검사_분할검사_비교.json",rows)\n        with (out/"전체검사_분할검사_비교.csv").open("w",encoding="utf-8-sig",newline="") as stream:\n            writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n        print("전체 검사·분할 검사 비교표를 같은 점수·IoU·이미지·정답 범위로 저장했습니다.",flush=True)\n    dump(work/"last_output.json",dict(paths=[str(p) for p in outputs],task="dataset",label=label,config=config))\n    return out\n\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser(description="SSPNet 통합 평가 처리")\n    parser.add_argument("action",choices=["prepare","baseline","run"])\n    parser.add_argument("--config",required=True)\n    parser.add_argument("--zip")\n    parser.add_argument("--work",default="/content/sspnet_free")\n    parser.add_argument("--store",required=True)\n    args=parser.parse_args()\n    config=json.loads(Path(args.config).read_text(encoding="utf-8"))\n    if args.action=="prepare":\n        prepare_dataset(config,args.work,args.zip)\n    elif args.action=="baseline":\n        import_baseline(args.zip,Path(args.store)/"baselines"/config["dataset_name"])\n        print("기존 전체 검사 결과를 준비했습니다. 다시 GPU로 추론하지 않습니다.",flush=True)\n    else:\n        run_dataset(config,args.work,args.store,Path(config.get("checkpoint") or Path(args.work)/"faster_rcnn_sspnet.pth"),os.environ.copy())\n'}
if not shutil.which("nvidia-smi"):
    raise RuntimeError("런타임 유형을 무료 GPU로 변경하세요.")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE=Path("/content/drive/MyDrive/SSPNet_free_colab")
else:
    STORE=WORK/"saved"
STORE.mkdir(parents=True,exist_ok=True)
HELPERS=WORK/"helpers"
HELPERS.mkdir(exist_ok=True)
for name,source in HELPER_SOURCES.items():
    (HELPERS/name).write_text(source,encoding="utf-8")
subprocess.run([sys.executable,str(HELPERS/"setup_environment.py")],check=True)
PYTHON=WORK/"venv/bin/python"
REPO=WORK/"SSPNet"
ENV=os.environ.copy()
ENV.pop("PYTHONHOME",None)
ENV.update(PYTHONNOUSERSITE="1",PYTHONPATH=str(REPO)+os.pathsep+str(HELPERS),MPLBACKEND="Agg")
def checked(command):
    subprocess.run([str(x) for x in command],check=True,env=ENV)
CHECKPOINT=WORK/"faster_rcnn_sspnet.pth"
WEIGHT_STORE=STORE/"weights/faster_rcnn_sspnet.pth"
WEIGHT_STORE.parent.mkdir(parents=True,exist_ok=True)
def valid_weight(path):
    return path.is_file() and path.stat().st_size>10_000_000
if WEIGHT_SOURCE == "public" and not valid_weight(CHECKPOINT):
    if valid_weight(WEIGHT_STORE):shutil.copy2(WEIGHT_STORE,CHECKPOINT)
    else:
        try:
            checked([PYTHON,"-c","import gdown; p=gdown.download(id='1IfPCt5xZqqBJ3sYVIuD5F9l29Jcy2Hn1',output=%r,quiet=False); assert p"%str(CHECKPOINT)])
        except subprocess.CalledProcessError:
            from google.colab import files
            print("가중치 자동 다운로드가 제한됐습니다. 저자의 Faster R-CNN SSPNet .pth 파일을 선택하세요.")
            uploaded=files.upload()
            if len(uploaded)!=1:raise ValueError("가중치 한 개를 선택하세요.")
            CHECKPOINT.write_bytes(next(iter(uploaded.values())))
if WEIGHT_SOURCE == "public":
    if not valid_weight(CHECKPOINT):raise ValueError("가중치 파일이 유효하지 않습니다.")
    import hashlib
    def weight_hash(path):
        h=hashlib.sha256()
        with open(path,"rb") as f:
            for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
        return h.hexdigest()
    if weight_hash(CHECKPOINT)!="912cc5b6f8a45b0f4521c9bf325868ed10dd4757497fb6ba8843fdbd26b09e4a":
        raise ValueError("공개 가중치 해시가 기준과 다릅니다. 기존 파일을 확인하세요.")
    if CHECKPOINT.resolve()!=WEIGHT_STORE.resolve():shutil.copy2(CHECKPOINT,WEIGHT_STORE)
elif WEIGHT_SOURCE == "trained_path":
    CHECKPOINT=Path(TRAINED_WEIGHT_PATH)
elif WEIGHT_SOURCE == "trained_upload":
    from google.colab import files
    print("추가 학습한 최종 .pth 가중치를 선택하세요.")
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError("가중치 한 개를 선택하세요.")
    CHECKPOINT=WORK/"trained_uploaded.pth"
    CHECKPOINT.write_bytes(next(iter(uploaded.values())))
else: raise ValueError("가중치 선택을 확인하세요.")
if not valid_weight(CHECKPOINT):raise ValueError("학습 가중치 경로 또는 파일을 확인하세요.")
CONFIG["checkpoint"]=str(CHECKPOINT)
CONFIG_PATH.write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2),encoding="utf-8")
shutil.copy2(WORK/"installed_packages.txt",STORE/"installed_packages.txt")
print("환경·가중치 준비 완료. 다음은 3단계입니다.")


## 3. 입력 준비

`dataset`이면 데이터 COCO ZIP을 선택합니다. 비교를 켰다면 기존 `SSPNet_all635_full_results.zip`도 선택합니다. 이미 준비된 파일은 재사용합니다. `single_image`이면 이미지 한 장을 선택합니다.


In [ ]:
#@title 3. 입력 준비
if not (WORK/"venv/bin/python").is_file() or "ENV" not in globals():
    raise RuntimeError("2단계를 먼저 실행하세요.")
def upload_one(prompt):
    from google.colab import files
    print(prompt)
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError("파일 한 개를 선택하세요.")
    return next(iter(uploaded.items()))
if CONFIG["task"]=="dataset":
    DATA_ZIP=Path(DATASET_ZIP_PATH) if DATASET_ZIP_PATH else WORK/(CONFIG["dataset_name"]+".zip")
    if not DATA_ZIP.is_file():
        _,content=upload_one("데이터셋 COCO JSON ZIP을 선택하세요.")
        DATA_ZIP=WORK/(CONFIG["dataset_name"]+".zip")
        DATA_ZIP.write_bytes(content)
    checked([PYTHON,HELPERS/"workflow.py","prepare","--config",CONFIG_PATH,"--zip",DATA_ZIP,"--store",STORE])
    baseline=STORE/"baselines"/CONFIG["dataset_name"]
    local_full=STORE/"evaluation"/CONFIG["dataset_name"]/CONFIG["split"]/"full"
    if CONFIG["compare_with_full"] and CONFIG["mode"]!="full":
        if BASELINE_ZIP_PATH or not any((p/"predictions_coco.json").is_file() for p in (baseline,local_full)):
            source=Path(BASELINE_ZIP_PATH) if BASELINE_ZIP_PATH else WORK/"baseline_full_results.zip"
            if not source.is_file():
                _,content=upload_one("이전에 다운로드한 전체 검사 결과 ZIP을 선택하세요.")
                source.write_bytes(content)
            checked([PYTHON,HELPERS/"workflow.py","baseline","--config",CONFIG_PATH,"--zip",source,"--store",STORE])
else:
    if IMAGE_SOURCE=="demo":IMAGE_PATH=REPO/"demo/demo.jpg"
    elif IMAGE_SOURCE=="path":IMAGE_PATH=Path(IMAGE_PATH_SETTING)
    else:
        name,content=upload_one("추론할 이미지 한 장을 선택하세요.")
        IMAGE_PATH=WORK/("input"+Path(name).suffix.lower())
        IMAGE_PATH.write_bytes(content)
    if not IMAGE_PATH.is_file():raise FileNotFoundError("이미지 파일이 없습니다.")
    display(DisplayImage(filename=str(IMAGE_PATH),width=800))
print("입력 준비 완료. 다음은 4단계입니다.")


## 4. 추론·평가

이 셀만 실제 모델 추론을 실행합니다. 사진마다 완료 기록을 저장합니다. 전체 사람·물속 사람 결과와 IoU 0.5 참고 결과를 같은 예측으로 집계합니다. 비교를 켰다면 기존 전체 검사 예측도 동일한 기준으로 다시 채점합니다.


In [ ]:
#@title 4. 추론·평가
if "ENV" not in globals() or not CHECKPOINT.is_file():raise RuntimeError("2~3단계를 먼저 실행하세요.")
if CONFIG["task"]=="dataset":
    checked([PYTHON,HELPERS/"workflow.py","run","--config",CONFIG_PATH,"--store",STORE])
else:
    if "IMAGE_PATH" not in globals() or not IMAGE_PATH.is_file():raise RuntimeError("3단계에서 이미지를 준비하세요.")
    settings=dict(repo=str(REPO),checkpoint=str(CHECKPOINT),mode=CONFIG["mode"],long_edge=CONFIG["long_edge"],short_edge=CONFIG["short_edge"],
                  tile_size=CONFIG["tile_size"],tile_overlap=CONFIG["tile_overlap"],merge_iou=.5,report_score=CONFIG["report_score"],
                  save_min_score=.001,max_detections=500,target_names=["person water"],match_iou=CONFIG["match_iou"])
    path=WORK/"single_settings.json"
    path.write_text(json.dumps(settings),encoding="utf-8")
    out=STORE/("single_image_"+CONFIG["mode"])
    checked([PYTHON,HELPERS/"engine.py","--settings",path,"--image",IMAGE_PATH,"--out",out])
    (WORK/"last_output.json").write_text(json.dumps(dict(task="single_image",paths=[str(out)],label=CONFIG["mode"],config=CONFIG)),encoding="utf-8")
print("실행 완료. 5단계에서 결과를 확인하고 다운로드하세요.")


## 5. 결과 확인·다운로드

전체 사람 결과가 주 결과입니다. 비교표의 재현율·정밀도는 0~1 비율입니다. 같은 모델·이미지·점수·IoU로 전체/분할을 비교합니다. 주석 누락이 있으면 실제 사람 탐지도 주석 기준 오탐에 포함될 수 있습니다. 현재 실행 결과와 비교 자료만 ZIP에 담습니다.


In [ ]:
#@title 5. 결과 확인·다운로드
if not (WORK/"last_output.json").is_file():raise RuntimeError("4단계를 먼저 실행하세요.")
last=json.loads((WORK/"last_output.json").read_text(encoding="utf-8"))
if last.get("config")!=CONFIG:raise RuntimeError("설정이 바뀌었습니다. 4단계를 실행한 뒤 결과를 확인하세요.")
paths=[Path(p) for p in last["paths"]]
if last["task"]=="dataset":
    out=paths[0]
    print((out/"전체사람_물속사람_결과요약.txt").read_text(encoding="utf-8"))
    import pandas as pd
    comparison=out/"전체검사_분할검사_비교.json"
    if CONFIG["compare_with_full"] and CONFIG["mode"]!="full" and comparison.is_file():
        print("전체 검사·분할 검사 비교표")
        display(pd.DataFrame(json.loads(comparison.read_text(encoding="utf-8"))))
else:
    display(DisplayImage(filename=str(paths[0]/"prediction.jpg"),width=1100))
result_zip=WORK/("SSPNet_"+last["task"]+"_"+last["label"]+"_results.zip")
with zipfile.ZipFile(result_zip,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in paths:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() not in (".pth",".pt",".tmp"):
                archive.write(path,path.relative_to(STORE))
    archive.write(CONFIG_PATH,"unified_settings.json")
    if (STORE/"installed_packages.txt").is_file():archive.write(STORE/"installed_packages.txt","installed_packages.txt")
print("결과 ZIP",result_zip)
from google.colab import files
files.download(str(result_zip))
